# Botel PMS — Property Dump Explorer

`dump_property.py` ile üretilen tek-property CSV'sini yükler ve pandas DataFrame'e çevirir. Aşağıdaki hücreleri sırayla çalıştır.

## CSV'nin yapısı
Her satır 4 kaynaktan birini temsil eder:

- **header** — bir konuşma thread'inin başlığı (`messageheader` tablosu)
- **message** — thread içindeki tek bir mesaj (`messageitem`)
- **task** — AI / manuel oluşturulmuş görev (`task`)
- **booking** — rezervasyon (`booking`)

`source` kolonu satırın hangi tabloya ait olduğunu söyler. `thread_id` ve `booking_id` satırları birbirine bağlar. `raw_json` orijinal kaydın tam JSON dump'ıdır — analizde sayısal/string olarak gelmemiş alanları oradan çekersin.


## 0) Yükle

In [ ]:
import json
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', 80)

CSV_PATH = Path('property_b23b29df_dump.csv')
df = pd.read_csv(
    CSV_PATH,
    parse_dates=['event_time', 'created_at'],
)
print(f'loaded {len(df):,} rows / {df.shape[1]} columns')
df.head(3)


## 1) Kolonlar ve veri tipleri
Toplam kaç kolon var ve her birinin tipi:


In [ ]:
df.dtypes


## 2) Kaynak tipi dağılımı

In [ ]:
df['source'].value_counts()


## 3) Zaman aralığı

In [ ]:
print('earliest :', df['event_time'].min())
print('latest   :', df['event_time'].max())
print('span     :', df['event_time'].max() - df['event_time'].min())


## 4) Kaynağa göre ayrılmış DataFrame'ler

Her source tipinin kendi DataFrame'ini ayır — boş kalan kolonlar (örn. message satırları için `amount`/`check_in`) anlam kazanmaz, kaynak başına bakmak daha temiz.


In [ ]:
df_header = df[df['source'] == 'header'].copy().reset_index(drop=True)
df_message = df[df['source'] == 'message'].copy().reset_index(drop=True)
df_task = df[df['source'] == 'task'].copy().reset_index(drop=True)
df_booking = df[df['source'] == 'booking'].copy().reset_index(drop=True)

print(f'headers : {len(df_header):>4}')
print(f'messages: {len(df_message):>4}')
print(f'tasks   : {len(df_task):>4}')
print(f'bookings: {len(df_booking):>4}')


## 5) Headers (thread'ler)

`title`: thread başlığı (genelde misafir adı). `status`: AI reply durumu (`pending` / `reply` / vb.). `channel`: kaynak kanalı.


In [ ]:
df_header[[
    'event_time', 'title', 'status', 'channel',
    'sentiment', 'check_in', 'check_out', 'booking_id',
]].head(10)


## 6) Messages (mesajlar)

`sender_or_actor` mesajı kimin gönderdiğini söyler (`guest` / `property` / agent kullanıcı id'si).


In [ ]:
print('sender breakdown:')
print(df_message['sender_or_actor'].value_counts().head(10))
print()
print('channel breakdown:')
print(df_message['channel'].value_counts())


In [ ]:
df_message[[
    'event_time', 'sender_or_actor', 'channel', 'title', 'body',
]].head(10)


## 7) Tasks (görevler)

AI veya manuel oluşturulmuş aksiyon kalemleri. `status` enum: `Pending`, `InProgress`, `Completed`, `Cancelled`, `Waiting`, `Monitoring`, `Done`.


In [ ]:
print('status breakdown:')
print(df_task['status'].value_counts())
print()
df_task[[
    'event_time', 'title', 'status', 'channel',
    'sender_or_actor', 'body',
]].head(15)


## 8) Bookings (rezervasyonlar)

`amount` MySQL'de `decimal(38,18)` — string olarak geliyor, sayısal kullanım için `to_numeric` ile çeviriyoruz.


In [ ]:
print('channel breakdown:')
print(df_booking['channel'].value_counts())
print()
print('status breakdown:')
print(df_booking['status'].value_counts())


In [ ]:
df_booking['amount_num'] = pd.to_numeric(
    df_booking['amount'], errors='coerce'
)
print('amount stats (EUR / native currency):')
df_booking['amount_num'].describe()


In [ ]:
df_booking[[
    'event_time', 'title', 'status', 'channel',
    'amount', 'check_in', 'check_out',
]].head(10)


## 9) raw_json — orijinal kayda iniş

Her satırın `raw_json` kolonu ilgili tablonun TÜM upstream alanlarını içerir.  Buradan `MessageItem`, `Task`, `Booking`, `MessageHeader` modelinin tam halini görebilirsin.


In [ ]:
# Bir mesajın tam upstream halini incele
sample = df_message.iloc[0]
parsed = json.loads(sample['raw_json'])
print(f'source = {sample["source"]}, fields = {len(parsed)}')
print()
for k, v in parsed.items():
    s = '' if v is None else str(v)
    print(f'  {k:30s} {s[:100] + "..." if len(s) > 100 else s}')


## 10) Thread → Messages bağlantısı

En çok mesajı olan thread'i seç, header'ını bul, sonra ona ait tüm mesajları zaman sırasıyla göster — bu past-conversation analiz için canonical view.


In [ ]:
counts = df_message['thread_id'].value_counts()
top_thread_id = counts.index[0]
print(f'top thread: {top_thread_id} '
      f'({counts.iloc[0]} mesaj)')

hdr_match = df_header[df_header['thread_id'] == top_thread_id]
if not hdr_match.empty:
    hdr = hdr_match.iloc[0]
    print(f'title  : {hdr["title"]!r}')
    print(f'channel: {hdr["channel"]}')
    print(f'booking: {hdr["booking_id"]}')


In [ ]:
df_message[df_message['thread_id'] == top_thread_id][[
    'event_time', 'sender_or_actor', 'body',
]].head(30)


## 11) Tek bir booking'in çevresi

Bir rezervasyona bağlı thread + mesajlar + task'lar — tek bir konuğun tüm temas geçmişi.


In [ ]:
# En yüksek tutarlı booking'i seç
top_booking = (
    df_booking.sort_values('amount_num', ascending=False).iloc[0]
)
bid = top_booking['booking_id']
print(f'booking : {top_booking["title"]} '
      f'({top_booking["channel"]}, '
      f'{top_booking["amount"]} {top_booking.get("currency", "")})')
print(f'check-in: {top_booking["check_in"]}')

linked = df[df['booking_id'] == bid]
print()
print(f'{len(linked)} satır bu booking\'e bağlı:')
linked[['event_time', 'source', 'sender_or_actor', 'title']].head(20)
